# Day 6 — Latency Benchmark
## SpatialMesh | Samsung AX Hackathon 2026

**Goal:** Measure end-to-end inference latency for the SpatialMesh pipeline.

**KPI Target:** Total ML inference < 20ms  
**Samsung Requirement:** Overall audio latency 40-60ms

**What we measure today:**
- CNN Encoder latency (built Day 5)
- HRTF Decoder latency (built Day 3)
- Full DSP pipeline latency
- Projected full pipeline with GRU + GNN estimates


In [4]:
# ─────────────────────────────────────────────
# Cell 1 — Imports
# ─────────────────────────────────────────────
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import os
import glob
import soundfile as sf
import sofar as sf_sofa
from scipy.signal import fftconvolve, resample
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print(f'All imports OK')

ImportError: cannot import name 'set_module' from 'pandas.util._decorators' (e:\Academics\BTP\Learning\.venv\Lib\site-packages\pandas\util\_decorators.py)

In [ ]:
# ─────────────────────────────────────────────
# Cell 2 — Config + Load CNN Model
# ─────────────────────────────────────────────

DRIVE_BASE   = '/content/drive/MyDrive/SpatialMesh'
MODEL_PATH   = os.path.join(DRIVE_BASE, 'models', 'cnn_encoder_best.pt')
SOFA_PATH    = os.path.join(DRIVE_BASE, 'sofa',
                            'P0001_FreeFieldComp_48kHz.sofa')
AUDIO_DIR    = os.path.join(DRIVE_BASE, 'audio')
OUTPUT_DIR   = os.path.join(DRIVE_BASE, 'benchmarks')
os.makedirs(OUTPUT_DIR, exist_ok=True)

SAMPLE_RATE  = 48000
CLIP_SAMPLES = 48000        # 1 second
EMBEDDING_DIM = 128
N_RUNS       = 200          # benchmark iterations for stable timing

# ── Rebuild CNN architecture (must match Day 5 v2) ──
class CNNEncoder(nn.Module):
    def __init__(self, embedding_dim=128, n_input_channels=2):
        super(CNNEncoder, self).__init__()
        self.backbone = nn.Sequential(
            nn.Conv1d(n_input_channels, 64,  kernel_size=64, stride=4, padding=32),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64,  128, kernel_size=32, stride=2, padding=16),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Conv1d(128, 256, kernel_size=16, stride=2, padding=8),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Conv1d(256, embedding_dim, kernel_size=8, stride=2, padding=4),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        features  = self.backbone(x)
        pooled    = self.pool(features)
        embedding = pooled.squeeze(-1)
        return F.normalize(embedding, p=2, dim=1)


# Load trained weights
cnn_model = CNNEncoder(embedding_dim=EMBEDDING_DIM, n_input_channels=2).to(DEVICE)
cnn_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
cnn_model.eval()

total_params = sum(p.numel() for p in cnn_model.parameters())
model_mb     = os.path.getsize(MODEL_PATH) / 1024 / 1024

print(f'CNN loaded from  : {MODEL_PATH}')
print(f'Parameters       : {total_params:,}')
print(f'Model size       : {model_mb:.2f} MB')

# Load SOFA
hrtf      = sf_sofa.read_sofa(SOFA_PATH)
positions = np.array(hrtf.SourcePosition)
ir        = np.array(hrtf.Data_IR)
fs_hrtf   = int(hrtf.Data_SamplingRate)
print(f'SOFA loaded      : {positions.shape[0]} positions, fs={fs_hrtf}Hz')

# Load one audio file for testing
audio_files = glob.glob(os.path.join(AUDIO_DIR, '*.flac'))
test_audio, audio_sr = sf.read(audio_files[0])
if test_audio.ndim == 2:
    test_audio = test_audio.mean(axis=1)
test_audio = test_audio.astype(np.float32)
if audio_sr != fs_hrtf:
    test_audio = resample(test_audio,
                          int(len(test_audio) * fs_hrtf / audio_sr)
                          ).astype(np.float32)
peak = np.abs(test_audio).max()
if peak > 0:
    test_audio /= peak
print(f'Test audio       : {len(test_audio)/fs_hrtf:.2f}s at {fs_hrtf}Hz')

In [ ]:
# ─────────────────────────────────────────────
# Cell 3 — Benchmark CNN Encoder
# Already measured in Day 5: ~2.35ms
# Re-measuring here for official benchmark record
# ─────────────────────────────────────────────

def benchmark_component(fn, n_runs=N_RUNS, warmup=10, label='Component'):
    """
    Benchmark a callable function.
    Runs warmup passes first (GPU needs warmup to reach full speed).
    Returns timing statistics in milliseconds.
    """
    # Warmup
    for _ in range(warmup):
        fn()
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()

    # Benchmark
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        fn()
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        end = time.perf_counter()
        times.append((end - start) * 1000)

    times = np.array(times)
    stats = {
        'label'  : label,
        'mean_ms': float(np.mean(times)),
        'med_ms' : float(np.median(times)),
        'p95_ms' : float(np.percentile(times, 95)),
        'p99_ms' : float(np.percentile(times, 99)),
        'min_ms' : float(np.min(times)),
        'max_ms' : float(np.max(times)),
        'std_ms' : float(np.std(times)),
    }
    return stats, times


# CNN input: stereo, 1 second
cnn_input = torch.randn(1, 2, CLIP_SAMPLES).to(DEVICE)

def run_cnn():
    with torch.no_grad():
        return cnn_model(cnn_input)

cnn_stats, cnn_times = benchmark_component(run_cnn, label='CNN Encoder')

print('CNN Encoder Benchmark')
print('=' * 45)
print(f'Mean   : {cnn_stats["mean_ms"]:6.3f} ms')
print(f'Median : {cnn_stats["med_ms"]:6.3f} ms')
print(f'P95    : {cnn_stats["p95_ms"]:6.3f} ms')
print(f'P99    : {cnn_stats["p99_ms"]:6.3f} ms')
print(f'Std    : {cnn_stats["std_ms"]:6.3f} ms')
print(f'Device : {DEVICE}')
print('=' * 45)
print(f'KPI (<3ms): {"✅ PASS" if cnn_stats["mean_ms"] < 3.0 else "⚠️  REVIEW"}')

In [ ]:
# ─────────────────────────────────────────────
# Cell 4 — Benchmark HRTF Decoder
# The fftconvolve operation from Day 3
# ─────────────────────────────────────────────

# Get HRIR for az=90° (standard test position)
az_rad  = np.radians(positions[:, 0])
el_rad  = np.radians(positions[:, 1])
tgt_az  = np.radians(90.0)
tgt_el  = np.radians(0.0)

distances = np.arccos(np.clip(
    np.sin(el_rad)*np.sin(tgt_el) +
    np.cos(el_rad)*np.cos(tgt_el)*np.cos(az_rad - tgt_az),
    -1.0, 1.0
))
idx    = int(np.argmin(distances))
hrir_L = ir[idx, 0, :] - np.mean(ir[idx, 0, :])
hrir_R = ir[idx, 1, :] - np.mean(ir[idx, 1, :])

# Prepare 1 second of test audio
audio_1s = test_audio[:CLIP_SAMPLES] if len(test_audio) >= CLIP_SAMPLES \
           else np.pad(test_audio, (0, CLIP_SAMPLES - len(test_audio)))

def run_hrtf_decoder():
    """
    HRTF Decoder = two fftconvolve operations
    One per ear (left + right)
    This is the core of the Day 3 pipeline
    """
    left  = fftconvolve(audio_1s, hrir_L, mode='full')
    right = fftconvolve(audio_1s, hrir_R, mode='full')
    peak  = max(np.abs(left).max(), np.abs(right).max())
    if peak > 0:
        left  /= peak
        right /= peak
    return np.stack([left[:CLIP_SAMPLES], right[:CLIP_SAMPLES]], axis=1)


hrtf_stats, hrtf_times = benchmark_component(
    run_hrtf_decoder, label='HRTF Decoder'
)

print('HRTF Decoder Benchmark')
print('=' * 45)
print(f'Mean   : {hrtf_stats["mean_ms"]:6.3f} ms')
print(f'Median : {hrtf_stats["med_ms"]:6.3f} ms')
print(f'P95    : {hrtf_stats["p95_ms"]:6.3f} ms')
print(f'HRIR length : {len(hrir_L)} samples ({len(hrir_L)/fs_hrtf*1000:.1f}ms)')
print(f'Audio length: {len(audio_1s)} samples (1000ms)')
print('=' * 45)
print(f'KPI (<3ms): {"✅ PASS" if hrtf_stats["mean_ms"] < 3.0 else "⚠️  REVIEW"}')

In [ ]:
# ─────────────────────────────────────────────
# Cell 5 — Benchmark 3-Speaker Pipeline
# CNN + HRTF Decoder for 3 concurrent speakers
# This is what actually runs in a real call
# ─────────────────────────────────────────────

# Prepare 3 speaker inputs
speaker_inputs = torch.randn(3, 2, CLIP_SAMPLES).to(DEVICE)

# 3 different positions (one per speaker)
speaker_positions = [
    (90,  0),   # Speaker A — LEFT
    (270, 0),   # Speaker B — RIGHT
    (0,   0),   # Speaker C — FRONT
]

# Precompute HRIRs for each position
speaker_hrirs = []
for az, el in speaker_positions:
    tgt_az = np.radians(az)
    tgt_el = np.radians(el)
    dists  = np.arccos(np.clip(
        np.sin(el_rad)*np.sin(tgt_el) +
        np.cos(el_rad)*np.cos(tgt_el)*np.cos(az_rad - tgt_az),
        -1.0, 1.0
    ))
    idx = int(np.argmin(dists))
    L   = ir[idx, 0, :] - np.mean(ir[idx, 0, :])
    R   = ir[idx, 1, :] - np.mean(ir[idx, 1, :])
    speaker_hrirs.append((L, R))


def run_3speaker_pipeline():
    """
    Full pipeline for 3 concurrent speakers:
    1. CNN encodes all 3 speakers simultaneously (batch)
    2. HRTF decoder renders each speaker at assigned position
    
    Note: GRU and GNN not yet built — benchmarked separately
    in Cell 6 using architecture-based estimates.
    """
    # Step 1: CNN encodes all 3 speakers in one batch forward pass
    with torch.no_grad():
        embeddings = cnn_model(speaker_inputs)   # (3, 128)

    # Step 2: HRTF decode each speaker at its position
    outputs = []
    for i, (hrir_L, hrir_R) in enumerate(speaker_hrirs):
        audio = audio_1s
        left  = fftconvolve(audio, hrir_L, mode='full')
        right = fftconvolve(audio, hrir_R, mode='full')
        outputs.append(np.stack(
            [left[:CLIP_SAMPLES], right[:CLIP_SAMPLES]], axis=1
        ))

    # Mix all speakers
    mixed = np.sum(outputs, axis=0) / len(outputs)
    return mixed, embeddings


pipeline_stats, pipeline_times = benchmark_component(
    run_3speaker_pipeline, label='3-Speaker Pipeline (CNN+HRTF)'
)

print('3-Speaker Pipeline Benchmark (CNN + HRTF)')
print('=' * 50)
print(f'Mean   : {pipeline_stats["mean_ms"]:6.3f} ms')
print(f'Median : {pipeline_stats["med_ms"]:6.3f} ms')
print(f'P95    : {pipeline_stats["p95_ms"]:6.3f} ms')
print(f'Speakers: 3 concurrent')
print('=' * 50)

In [ ]:
# ─────────────────────────────────────────────
# Cell 6 — Full Pipeline Latency Projection
# GRU and GNN not yet built — estimated from architecture
# ─────────────────────────────────────────────

# ── GRU Tracker estimate ──
# Architecture: GRU(128, 64, num_layers=2) on 10 frames
# Very small model — 2 layers, hidden=64
# Similar models on T4: typically 0.5-1.5ms
GRU_ESTIMATE_MS = 1.0

# ── GAT-GNN estimate ──
# Architecture: 2-layer GAT, 4 attention heads
# Graph: 3 nodes, 6 edges (tiny graph)
# Similar models on T4: typically 3-6ms
GNN_ESTIMATE_MS = 5.0

# ── Measured components ──
CNN_MEASURED_MS  = cnn_stats['mean_ms']
HRTF_MEASURED_MS = hrtf_stats['mean_ms']

# ── Total projection ──
TOTAL_PROJECTED_MS = (
    CNN_MEASURED_MS  +
    GRU_ESTIMATE_MS  +
    GNN_ESTIMATE_MS  +
    HRTF_MEASURED_MS
)

print('Full Pipeline Latency Projection')
print('=' * 55)
print(f'{"Component":<25} {"Latency":>10}  {"Method"}')
print('-' * 55)
print(f'{"CNN Encoder":<25} {CNN_MEASURED_MS:>9.2f}ms  Measured (200 runs, T4 GPU)')
print(f'{"GRU Tracker":<25} {GRU_ESTIMATE_MS:>9.2f}ms  Estimated (2-layer, hidden=64)')
print(f'{"GAT-GNN":<25} {GNN_ESTIMATE_MS:>9.2f}ms  Estimated (3 nodes, 2-layer, 4-head)')
print(f'{"HRTF Decoder":<25} {HRTF_MEASURED_MS:>9.2f}ms  Measured (200 runs, fftconvolve)')
print('-' * 55)
print(f'{"TOTAL":<25} {TOTAL_PROJECTED_MS:>9.2f}ms')
print('=' * 55)
print(f'KPI (<20ms): {"✅ PASS" if TOTAL_PROJECTED_MS < 20.0 else "⚠️  REVIEW"}')
print(f'\nNote: GRU and GNN estimates will be replaced with')
print(f'measured values after Day 10 and Day 11.')

In [ ]:
# ─────────────────────────────────────────────
# Cell 7 — Samsung Audio Latency Budget
# Shows SpatialMesh fits within 40-60ms requirement
# ─────────────────────────────────────────────

print('Samsung AX Requirement: 40-60ms total audio latency')
print('=' * 55)
print(f'{"Stage":<30} {"Latency":>10}  {"Notes"}')
print('-' * 55)

stages = [
    ('Audio capture (mic buffer)',   10.0, 'Standard WebRTC'),
    ('Network transport',            15.0, 'WebRTC typical LAN'),
    ('SpatialMesh ML pipeline',  TOTAL_PROJECTED_MS, 'Our system'),
    ('Audio playback (speaker)',      7.0, 'Standard buffer'),
]

total_system = 0
for stage, ms, note in stages:
    print(f'{stage:<30} {ms:>9.1f}ms  {note}')
    total_system += ms

print('-' * 55)
print(f'{"TOTAL SYSTEM LATENCY":<30} {total_system:>9.1f}ms')
print('=' * 55)

samsung_ok = 40 <= total_system <= 60
print(f'Samsung requirement (40-60ms): '
      f'{"✅ PASS" if samsung_ok else "⚠️  REVIEW"}')
print(f'\nSpatialMesh ML pipeline uses only {TOTAL_PROJECTED_MS/total_system*100:.1f}%')
print(f'of the total system latency budget.')

In [ ]:
# ─────────────────────────────────────────────
# Cell 8 — Latency Breakdown Plot
# Visual proof for documentation and YouTube demo
# ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('SpatialMesh — Latency Analysis\nSamsung AX Hackathon 2026',
             fontsize=13, fontweight='bold')

# ── Plot 1: ML Pipeline breakdown ──
ax1 = axes[0]
components = ['CNN\nEncoder', 'GRU\nTracker', 'GAT-GNN\nSpatial Core', 'HRTF\nDecoder']
latencies  = [CNN_MEASURED_MS, GRU_ESTIMATE_MS, GNN_ESTIMATE_MS, HRTF_MEASURED_MS]
colors     = ['#2196F3', '#FF9800', '#F44336', '#4CAF50']
methods    = ['Measured', 'Estimated', 'Estimated', 'Measured']
hatches    = ['', '///', '///', '']

bars = ax1.bar(components, latencies, color=colors,
               edgecolor='white', linewidth=1.5)

# Add hatch for estimated components
for bar, hatch in zip(bars, hatches):
    bar.set_hatch(hatch)

# Add value labels on bars
for bar, val, method in zip(bars, latencies, methods):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{val:.1f}ms\n({method})',
             ha='center', va='bottom', fontsize=9, fontweight='bold')

ax1.axhline(20, color='red', linestyle='--', linewidth=1.5,
            label='20ms KPI limit')
ax1.text(3.4, 20.3, 'KPI limit', color='red', fontsize=9)
ax1.set_ylabel('Latency (ms)', fontsize=11)
ax1.set_title(f'ML Pipeline Breakdown\nTotal: {TOTAL_PROJECTED_MS:.1f}ms',
              fontsize=11)
ax1.set_ylim(0, 22)
ax1.grid(True, alpha=0.3, axis='y')

# Legend for hatch
measured_patch  = mpatches.Patch(color='grey', label='Measured')
estimated_patch = mpatches.Patch(color='grey', hatch='///', label='Estimated')
ax1.legend(handles=[measured_patch, estimated_patch], fontsize=9)

# ── Plot 2: Full system latency budget ──
ax2 = axes[1]
system_labels = [
    'Audio\nCapture', 'Network\nTransport',
    'SpatialMesh\nML Pipeline', 'Audio\nPlayback'
]
system_vals   = [10.0, 15.0, TOTAL_PROJECTED_MS, 7.0]
system_colors = ['#9E9E9E', '#9E9E9E', '#2196F3', '#9E9E9E']

bars2 = ax2.bar(system_labels, system_vals, color=system_colors,
                edgecolor='white', linewidth=1.5)

for bar, val in zip(bars2, system_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}ms', ha='center', va='bottom',
             fontsize=10, fontweight='bold')

# Samsung requirement band
ax2.axhline(total_system, color='blue', linestyle='-',
            linewidth=2, label=f'Total: {total_system:.1f}ms')
ax2.axhspan(40, 60, alpha=0.1, color='green', label='Samsung 40-60ms target')
ax2.set_ylabel('Latency (ms)', fontsize=11)
ax2.set_title(f'Full System Latency Budget\n'
              f'Total: {total_system:.1f}ms | Samsung target: 40-60ms',
              fontsize=11)
ax2.set_ylim(0, 65)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, 'latency_breakdown.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {plot_path}')

In [ ]:
# ─────────────────────────────────────────────
# Cell 9 — Save Benchmark Results to CSV
# Official record for documentation
# ─────────────────────────────────────────────

benchmark_data = [
    {
        'component'  : 'CNN Encoder',
        'mean_ms'    : cnn_stats['mean_ms'],
        'median_ms'  : cnn_stats['med_ms'],
        'p95_ms'     : cnn_stats['p95_ms'],
        'p99_ms'     : cnn_stats['p99_ms'],
        'method'     : 'Measured',
        'device'     : str(DEVICE),
        'n_runs'     : N_RUNS,
        'kpi_target' : 3.0,
        'kpi_pass'   : cnn_stats['mean_ms'] < 3.0,
    },
    {
        'component'  : 'GRU Tracker',
        'mean_ms'    : GRU_ESTIMATE_MS,
        'median_ms'  : GRU_ESTIMATE_MS,
        'p95_ms'     : GRU_ESTIMATE_MS * 1.5,
        'p99_ms'     : GRU_ESTIMATE_MS * 2.0,
        'method'     : 'Estimated (architecture-based)',
        'device'     : str(DEVICE),
        'n_runs'     : 0,
        'kpi_target' : 4.0,
        'kpi_pass'   : GRU_ESTIMATE_MS < 4.0,
    },
    {
        'component'  : 'GAT-GNN Spatial Core',
        'mean_ms'    : GNN_ESTIMATE_MS,
        'median_ms'  : GNN_ESTIMATE_MS,
        'p95_ms'     : GNN_ESTIMATE_MS * 1.5,
        'p99_ms'     : GNN_ESTIMATE_MS * 2.0,
        'method'     : 'Estimated (architecture-based)',
        'device'     : str(DEVICE),
        'n_runs'     : 0,
        'kpi_target' : 8.0,
        'kpi_pass'   : GNN_ESTIMATE_MS < 8.0,
    },
    {
        'component'  : 'HRTF Decoder',
        'mean_ms'    : hrtf_stats['mean_ms'],
        'median_ms'  : hrtf_stats['med_ms'],
        'p95_ms'     : hrtf_stats['p95_ms'],
        'p99_ms'     : hrtf_stats['p99_ms'],
        'method'     : 'Measured',
        'device'     : 'CPU (scipy fftconvolve)',
        'n_runs'     : N_RUNS,
        'kpi_target' : 3.0,
        'kpi_pass'   : hrtf_stats['mean_ms'] < 3.0,
    },
    {
        'component'  : 'TOTAL PIPELINE',
        'mean_ms'    : TOTAL_PROJECTED_MS,
        'median_ms'  : TOTAL_PROJECTED_MS,
        'p95_ms'     : TOTAL_PROJECTED_MS * 1.2,
        'p99_ms'     : TOTAL_PROJECTED_MS * 1.4,
        'method'     : 'Measured + Estimated',
        'device'     : 'GPU + CPU',
        'n_runs'     : N_RUNS,
        'kpi_target' : 20.0,
        'kpi_pass'   : TOTAL_PROJECTED_MS < 20.0,
    },
]

df = pd.DataFrame(benchmark_data)
csv_path = os.path.join(OUTPUT_DIR, 'latency_benchmark.csv')
df.to_csv(csv_path, index=False)

print(f'Benchmark results saved → {csv_path}')
print(f'\n{df[["component", "mean_ms", "method", "kpi_pass"]].to_string()}')

In [ ]:
# ─────────────────────────────────────────────
# Cell 10 — Day 6 Summary
# ─────────────────────────────────────────────

print('=' * 60)
print('SPATIALMESH — DAY 6 BENCHMARK SUMMARY')
print('=' * 60)
print(f'\nML Pipeline Components:')
print(f'  CNN Encoder      : {CNN_MEASURED_MS:.2f}ms  (MEASURED ✅)')
print(f'  GRU Tracker      : {GRU_ESTIMATE_MS:.2f}ms  (estimated)')
print(f'  GAT-GNN          : {GNN_ESTIMATE_MS:.2f}ms  (estimated)')
print(f'  HRTF Decoder     : {HRTF_MEASURED_MS:.2f}ms  (MEASURED ✅)')
print(f'  ─────────────────────────────')
print(f'  Total projected  : {TOTAL_PROJECTED_MS:.2f}ms')
print(f'  KPI (<20ms)      : {"✅ PASS" if TOTAL_PROJECTED_MS < 20 else "⚠️  REVIEW"}')
print(f'\nSystem Latency Budget:')
print(f'  Audio capture    : 10.0ms')
print(f'  Network          : 15.0ms')
print(f'  SpatialMesh ML   : {TOTAL_PROJECTED_MS:.1f}ms')
print(f'  Audio playback   :  7.0ms')
print(f'  ─────────────────────────────')
print(f'  Total system     : {total_system:.1f}ms')
print(f'  Samsung (40-60ms): {"✅ PASS" if 40 <= total_system <= 60 else "⚠️  REVIEW"}')
print(f'\nFiles saved:')
print(f'  {csv_path}')
print(f'  {plot_path}')
print('=' * 60)
print('Next: Day 7 — Graph Builder')
print('  Build speaker graph: nodes=speakers, edges=spatial relationships')